<a href="https://colab.research.google.com/github/Ramank510/CMP7005--Beijing-Air-Quality-Analysis/blob/main/CMP7005_PRAC1_S2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CMP7005 — Programming for Data Analysis
## PRAC1
### Beijing Air Quality Analysis (2013–2017)
---
**Module:** CMP7005 | **Semester:** 2 | **Year:** 2025–2026

**Stations selected:** Dongsi, Guanyuan (Urban) | Dingling, Huairou (Suburban)

---

### Installing the libraries

In [2]:
# Install Streamlit (needed for the app in Task 4)
!pip install -q streamlit

# Import all required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import os
import pickle
import zipfile

# Machine learning
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Settings
warnings.filterwarnings('ignore')
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['axes.titlesize'] = 14
sns.set_style('whitegrid')

# Create folders
os.makedirs('data', exist_ok=True)
os.makedirs('figures', exist_ok=True)

print('✅ All libraries imported!')
print('✅ Folders created!')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 61.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 96.7 MB/s eta 0:00:00
✅ All libraries imported!
✅ Folders created!


---
## Uploading the  Data


In [3]:
# Upload all the datasets (zip file)
from google.colab import files

uploaded = files.upload()

for filename in uploaded.keys():
    os.rename(filename, f'data/{filename}')
    print(f'✅ Uploaded: {filename}')

Saving Assessment Data-20260306.zip to Assessment Data-20260306.zip
✅ Uploaded: Assessment Data-20260306.zip


In [4]:
# Extract the zip file
with zipfile.ZipFile('data/Assessment Data-20260306.zip', 'r') as zip_ref:
    zip_ref.extractall('data/')

print('✅ Zip extracted!')
print('\nCSV files available:')
for f in sorted(os.listdir('data/')):
    if f.endswith('.csv'):
        print(f'  - {f}')

✅ Zip extracted!

CSV files available:
  - PRSA_Data_Aotizhongxin_20130301-20170228.csv
  - PRSA_Data_Changping_20130301-20170228.csv
  - PRSA_Data_Dingling_20130301-20170228.csv
  - PRSA_Data_Dongsi_20130301-20170228.csv
  - PRSA_Data_Guanyuan_20130301-20170228.csv
  - PRSA_Data_Gucheng_20130301-20170228.csv
  - PRSA_Data_Huairou_20130301-20170228.csv
  - PRSA_Data_Nongzhanguan_20130301-20170228.csv
  - PRSA_Data_Shunyi_20130301-20170228.csv
  - PRSA_Data_Tiantan_20130301-20170228.csv
  - PRSA_Data_Wanliu_20130301-20170228.csv
  - PRSA_Data_Wanshouxigong_20130301-20170228.csv


---
## 1. Data Selection & Handling


### Station Selection Justification
Four stations were selected based on Xu & Zhang (2004) and Yao et al. (2015):

| Station | Type | Reason |
|---------|------|--------|
| Dongsi | Urban | Central Beijing, Dongcheng district — high traffic, high population density |
| Guanyuan | Urban | Xicheng district, inner city — near major roads and commercial activity |
| Dingling | Suburban | Changping district, north Beijing — semi-rural, near Ming Tombs |
| Huairou | Suburban | Huairou district, northernmost — mountainous, semi-rural, low industrial activity |

In [5]:
# Load 4 selected station CSV files
selected_stations = ['Dongsi', 'Guanyuan', 'Dingling', 'Huairou']
dataframes = []

for station in selected_stations:
    filepath = f'data/PRSA_Data_{station}_20130301-20170228.csv'
    df = pd.read_csv(filepath)
    print(f'✅ Loaded {station}: {df.shape[0]:,} rows, {df.shape[1]} columns')
    dataframes.append(df)

# Merge into one combined dataset
combined_df = pd.concat(dataframes, ignore_index=True)

# Create datetime column
combined_df['datetime'] = pd.to_datetime(combined_df[['year', 'month', 'day', 'hour']])

# Save combined dataset
combined_df.to_csv('data/combined_dataset.csv', index=False)

print(f'\n✅ Combined dataset: {combined_df.shape[0]:,} rows x {combined_df.shape[1]} columns')
print(f'📅 Date range: {combined_df["datetime"].min().date()} to {combined_df["datetime"].max().date()}')
print(f'🏙️  Stations: {list(combined_df["station"].unique())}')

✅ Loaded Dongsi: 35,064 rows, 18 columns
✅ Loaded Guanyuan: 35,064 rows, 18 columns
✅ Loaded Dingling: 35,064 rows, 18 columns
✅ Loaded Huairou: 35,064 rows, 18 columns

✅ Combined dataset: 140,256 rows x 19 columns
📅 Date range: 2013-03-01 to 2017-02-28
🏙️  Stations: ['Dongsi', 'Guanyuan', 'Dingling', 'Huairou']
